# Article NER Service — Jalankan Aplikasi

Notebook ini menyiapkan aplikasi web dari source code dan model yang ada di repository. Tidak perlu mengunggah CSV dataset dari dosen.

Jalankan sel satu per satu dari atas ke bawah. Setelah tautan Cloudflare muncul, buka tautan itu untuk memakai aplikasi. Kolom judul dan isi artikel dimulai dalam keadaan kosong.

## 1. Unduh source code dan model dari repository

In [ ]:
from pathlib import Path
import io
import os
import re
import sys
import json
import time
import zipfile
import shutil
import subprocess
import urllib.request
import requests
from IPython.display import display, HTML

REPO_RAW = "https://raw.githubusercontent.com/DheandraKhairunnisaP/Data-Science-UTS/main"
PROJECT = Path("/content/article_ner_service")

if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)

# Unduh source code aplikasi langsung dari repository.
source_files = [
    "app/main.py",
    "app/__init__.py",
    "web/index.html",
    "tests/test_api.py",
    "requirements.txt",
]
for relative_path in source_files:
    response = requests.get(f"{REPO_RAW}/{relative_path}", timeout=60)
    response.raise_for_status()
    destination = PROJECT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(response.content)
    print("OK:", relative_path)

# Unduh model yang dipakai oleh aplikasi, lalu periksa arsip sebelum diekstrak.
model_url = f"{REPO_RAW}/models/ner_custom_v1.zip"
model_response = requests.get(model_url, timeout=180)
model_response.raise_for_status()

model_zip = io.BytesIO(model_response.content)
with zipfile.ZipFile(model_zip) as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise RuntimeError(f"Arsip model rusak pada file: {bad_member}")
    if "config.cfg" not in archive.namelist():
        raise FileNotFoundError("config.cfg tidak ditemukan di root ZIP model.")
    model_dir = PROJECT / "models" / "ner_custom_v1"
    model_dir.mkdir(parents=True, exist_ok=True)
    archive.extractall(model_dir)

assert (model_dir / "config.cfg").is_file()
print(f"Model berhasil disiapkan ({len(model_response.content):,} bytes).")
print("Folder project:", PROJECT)

## 2. Pasang dependensi

In [ ]:
install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")],
    text=True, capture_output=True
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr[-6000:])
    raise RuntimeError("Instalasi dependensi gagal.")
print("Dependensi berhasil dipasang.")

## 3. Jalankan pengujian API

In [ ]:
test_run = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", str(PROJECT / "tests")],
    cwd=str(PROJECT), text=True, capture_output=True
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr[-2000:])
if test_run.returncode != 0:
    raise RuntimeError("Ada pengujian yang gagal. Periksa hasil di atas sebelum melanjutkan.")
print("Semua pengujian lulus.")

## 4. Jalankan backend

In [ ]:
# Hentikan server yang mungkin masih berjalan dari eksekusi sebelumnya.
if "server" in globals() and server is not None and server.poll() is None:
    server.terminate()
    try:
        server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server.kill()

server_log_path = Path("/content/article_ner_uvicorn.log")
server_log = open(server_log_path, "w", encoding="utf-8")
server = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn", "app.main:app",
        "--host", "0.0.0.0", "--port", "8000"
    ],
    cwd=str(PROJECT),
    stdout=server_log,
    stderr=subprocess.STDOUT
)

health = None
for _ in range(45):
    if server.poll() is not None:
        break
    try:
        result = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if result.status_code == 200:
            health = result.json()
            break
    except requests.RequestException:
        pass
    time.sleep(1)

if health is None:
    server_log.flush()
    print(server_log_path.read_text(encoding="utf-8")[-6000:])
    raise RuntimeError("Backend tidak aktif. Periksa log di atas.")

print("Backend aktif:", health)

## 5. Buat tautan aplikasi

In [ ]:
cloudflared = "/content/cloudflared"
if not Path(cloudflared).exists():
    print("Mengunduh Cloudflare Tunnel...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared
    )
    os.chmod(cloudflared, 0o755)

if "cloudflared_process" in globals() and cloudflared_process is not None:
    if cloudflared_process.poll() is None:
        cloudflared_process.terminate()
        try:
            cloudflared_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            cloudflared_process.kill()

tunnel_log_path = Path("/content/article_ner_tunnel.log")
tunnel_log = open(tunnel_log_path, "w", encoding="utf-8")
cloudflared_process = subprocess.Popen(
    [cloudflared, "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

public_url = None
for _ in range(60):
    if tunnel_log_path.exists():
        log_text = tunnel_log_path.read_text(encoding="utf-8", errors="replace")
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            public_url = match.group(0)
            break
    if cloudflared_process.poll() is not None:
        break
    time.sleep(1)

if not public_url:
    tunnel_log.flush()
    print(tunnel_log_path.read_text(encoding="utf-8", errors="replace")[-6000:])
    raise RuntimeError("Tautan publik gagal dibuat. Backend tetap tersedia di port 8000 pada runtime.")

print("Tautan aplikasi:", public_url)

# Periksa URL publik terlebih dahulu.
public_health = requests.get(public_url + "/health", timeout=30)
print("Status health publik:", public_health.status_code)
if public_health.status_code != 200:
    print(public_health.text[:1000])
    raise RuntimeError("Tautan dibuat, tetapi pemeriksaan health publik gagal.")

display(HTML(f"""
<div style="border:1px solid #dbe3ef;border-radius:12px;padding:18px;font-family:Arial,sans-serif">
  <h2 style="margin-top:0">Article NER Service</h2>
  <p>Backend aktif dan tautan sudah diperiksa.</p>
  <a href="{public_url}" target="_blank" rel="noopener"
     style="display:inline-block;background:#2855a6;color:white;padding:12px 18px;border-radius:7px;text-decoration:none;font-weight:bold">
    Buka aplikasi
  </a>
  <p style="margin-bottom:0"><a href="{public_url}/docs" target="_blank" rel="noopener">Dokumentasi API</a></p>
  <p style="color:#667085;font-size:12px">Biarkan runtime Colab dan proses tunnel tetap berjalan.</p>
</div>
"""))

## 6. Periksa request contoh

In [ ]:
payload = {
    "title": "Malaysia and Maybank Announce New Plans",
    "body": "Maybank said the company would expand its services in Kuala Lumpur on October 22, 2026."
}
response = requests.post(
    "http://127.0.0.1:8000/extract-entities",
    json=payload,
    timeout=30
)
print("HTTP status:", response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))
response.raise_for_status()
combined_text = payload["title"] + "\n" + payload["body"]
for entity in response.json()["entities"]:
    assert combined_text[entity["start"]:entity["end"]] == entity["text"]
print("Output dan posisi karakter berhasil diperiksa.")

## Menggunakan aplikasi

1. Klik tautan aplikasi dari sel 5.
2. Isi judul dan isi artikel pada halaman.
3. Klik **Analisis Entitas**.
4. Periksa tabel entitas dan bagian teks yang disorot.

Kolom input dimulai kosong. Tombol **Isi Contoh** tersedia jika ingin mencoba artikel contoh. URL Cloudflare bersifat sementara dan akan berhenti berfungsi jika runtime atau proses tunnel dihentikan.